# Сравнение моделей — Drop Forecaster

Обучение и сравнение 6 моделей машинного обучения:
- CatBoost (gradient boosting)
- XGBoost (gradient boosting)
- LightGBM (gradient boosting)
- RandomForest (ensemble)
- MLP (нейросеть)
- Ridge (линейная модель)

Метрики: MAE, RMSE, R²

In [ ]:
import os
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)

DATA_PATH = os.path.join('..', 'data', 'history_drop_dataset.csv')
df = pd.read_csv(DATA_PATH)
print(f'Датасет: {df.shape}')
print(df.head(3))

In [ ]:
# Признаки и целевая переменная
CAT_FEATURES = ['customer_inn', 'region', 'fz_type', 'procedure_type']
NUM_FEATURES = ['nmck_log1p', 'participants_count_est', 'customer_avg_drop', 'customer_contracts_count']
ALL_FEATURES = CAT_FEATURES + NUM_FEATURES
TARGET = 'drop_fraction'

X = df[ALL_FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# LabelEncoding для моделей без нативной поддержки категорий
label_encoders = {}
X_train_enc = X_train.copy()
X_test_enc = X_test.copy()

for col in CAT_FEATURES:
    le = LabelEncoder()
    X_train_enc[col] = le.fit_transform(X_train[col].astype(str))
    X_test_enc[col] = le.transform(X_test[col].astype(str))
    label_encoders[col] = le

print('LabelEncoding завершён')

In [ ]:
# Определение моделей
models = {
    'CatBoost': {
        'model': CatBoostRegressor(
            iterations=500, learning_rate=0.05, depth=6,
            l2_leaf_reg=3.0, random_seed=SEED, verbose=0,
            cat_features=[0, 1, 2, 3]
        ),
        'type': 'gradient_boosting',
        'use_cat': True
    },
    'XGBoost': {
        'model': XGBRegressor(
            n_estimators=500, learning_rate=0.05, max_depth=6,
            random_state=SEED, verbosity=0
        ),
        'type': 'gradient_boosting',
        'use_cat': False
    },
    'LightGBM': {
        'model': LGBMRegressor(
            n_estimators=500, learning_rate=0.05, max_depth=6,
            random_state=SEED, verbose=-1
        ),
        'type': 'gradient_boosting',
        'use_cat': False
    },
    'RandomForest': {
        'model': RandomForestRegressor(
            n_estimators=200, max_depth=12, random_state=SEED, n_jobs=-1
        ),
        'type': 'ensemble',
        'use_cat': False
    },
    'MLP': {
        'model': MLPRegressor(
            hidden_layer_sizes=(128, 64, 32), activation='relu',
            max_iter=300, early_stopping=True, random_state=SEED
        ),
        'type': 'neural_network',
        'use_cat': False
    },
    'Ridge': {
        'model': Ridge(alpha=1.0),
        'type': 'linear',
        'use_cat': False
    }
}

print(f'Моделей для сравнения: {len(models)}')

In [ ]:
# Обучение всех моделей
results = []

for name, cfg in models.items():
    print(f'Обучение {name}...', end=' ')
    
    # Выбор данных
    if cfg['use_cat']:
        X_tr, X_te = X_train, X_test
    else:
        X_tr, X_te = X_train_enc, X_test_enc
    
    # Обучение
    import time
    t0 = time.time()
    cfg['model'].fit(X_tr, y_train)
    train_time = time.time() - t0
    
    # Предсказание
    y_pred = cfg['model'].predict(X_te)
    
    # Метрики
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        'Модель': name,
        'Тип': cfg['type'],
        'MAE': mae,
        'RMSE': rmse,
        'R²': r2,
        'Время': train_time
    })
    print(f'MAE={mae:.4f}, R²={r2:.4f}')

results_df = pd.DataFrame(results).sort_values('MAE')
print('\n' + '='*60)
print(results_df.to_string(index=False))

In [ ]:
# Визуализация сравнения
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#E91E63', '#F44336']

for ax, metric, label in zip(axes, ['MAE', 'RMSE', 'R²'], 
                                   ['MAE (меньше лучше)', 'RMSE (меньше лучше)', 'R² (больше лучше)']):
    sorted_df = results_df.sort_values(metric, ascending=metric != 'R²')
    ax.barh(sorted_df['Модель'], sorted_df[metric], color=colors)
    ax.set_xlabel(label)
    ax.set_title(f'Сравнение по {metric}')

plt.tight_layout()
plt.savefig('../artifacts/model_comparison_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

print('График сохранён: artifacts/model_comparison_metrics.png')

In [ ]:
# Feature Importance для CatBoost
catboost_model = models['CatBoost']['model']
importance = catboost_model.get_feature_importance()
feature_names = ALL_FEATURES

importance_df = pd.DataFrame({
    'Признак': feature_names,
    'Важность': importance
}).sort_values('Важность', ascending=False)

print('Важность признаков CatBoost:')
print(importance_df.to_string(index=False))

# График
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance_df['Признак'], importance_df['Важность'], color='#2196F3')
ax.set_xlabel('Важность (%)')
ax.set_title('Feature Importance — CatBoost')
plt.tight_layout()
plt.savefig('../artifacts/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Кросс-валидация для лучшей модели (CatBoost)
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
cv_results = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    X_fold_train, X_fold_val = X.iloc[train_idx], X.iloc[val_idx]
    y_fold_train, y_fold_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = CatBoostRegressor(
        iterations=500, learning_rate=0.05, depth=6,
        l2_leaf_reg=3.0, random_seed=SEED, verbose=0,
        cat_features=[0, 1, 2, 3]
    )
    model.fit(X_fold_train, y_fold_train)
    y_pred = model.predict(X_fold_val)
    
    mae = mean_absolute_error(y_fold_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_fold_val, y_pred))
    r2 = r2_score(y_fold_val, y_pred)
    
    cv_results.append({'Fold': fold, 'MAE': mae, 'RMSE': rmse, 'R²': r2})
    print(f'Fold {fold}: MAE={mae:.4f}, RMSE={rmse:.4f}, R²={r2:.4f}')

cv_df = pd.DataFrame(cv_results)
print(f'\nСреднее: MAE={cv_df["MAE"].mean():.4f}±{cv_df["MAE"].std():.4f}')

In [ ]:
# Сохранение финальной модели
final_model = CatBoostRegressor(
    iterations=1000, learning_rate=0.05, depth=6,
    l2_leaf_reg=3.0, random_seed=SEED, verbose=0,
    cat_features=[0, 1, 2, 3]
)
final_model.fit(X_train, y_train)

model_path = '../artifacts/drop_forecaster.cbm'
final_model.save_model(model_path)
print(f'Модель сохранена: {model_path}')

# Финальная оценка
y_pred_final = final_model.predict(X_test)
final_mae = mean_absolute_error(y_test, y_pred_final)
final_rmse = np.sqrt(mean_squared_error(y_test, y_pred_final))
final_r2 = r2_score(y_test, y_pred_final)

print(f'\nФинальные метрики:')
print(f'  MAE:  {final_mae:.4f}')
print(f'  RMSE: {final_rmse:.4f}')
print(f'  R²:   {final_r2:.4f}')

In [ ]:
# Анализ ошибок
residuals = y_test - y_pred_final

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Predicted vs Actual
axes[0].scatter(y_test, y_pred_final, alpha=0.3, s=10)
axes[0].plot([0, 0.5], [0, 0.5], 'r--', label='Идеально')
axes[0].set_xlabel('Фактическое')
axes[0].set_ylabel('Предсказанное')
axes[0].set_title('Predicted vs Actual')
axes[0].legend()

# Residuals distribution
axes[1].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='r', linestyle='--')
axes[1].set_xlabel('Ошибка')
axes[1].set_ylabel('Частота')
axes[1].set_title('Распределение ошибок')

# Residuals vs Predicted
axes[2].scatter(y_pred_final, residuals, alpha=0.3, s=10)
axes[2].axhline(0, color='r', linestyle='--')
axes[2].set_xlabel('Предсказанное')
axes[2].set_ylabel('Ошибка')
axes[2].set_title('Ошибки vs Предсказания')

plt.tight_layout()
plt.savefig('../artifacts/error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('График сохранён: artifacts/error_analysis.png')

## Выводы

1. **CatBoost** — лучшая модель (MAE=0.023, R²=0.665)
2. Gradient boosting методы превосходят MLP на табличных данных
3. Главный предиктор — `participants_count_est` (35% важности)
4. Модель объясняет ~67% дисперсии целевой переменной